# Lista 01 - CKP8366 - TÓPICOS ESPECIAIS EM LÓGICA I

Realizado por Gustavo Fernandez Vidal Vazquez, matrícula 603150. 

In [1]:
from pysat.formula import *

## Tarefa 1 - Escreva a função (recursiva) subFormula(f) que retorna todas as subfórmulas de uma fórmula.
Dica: podemos saber se uma fórmula f é atômica (do tipo Atom) usando a
função booleana isinstance(f,Atom), para saber se uma fórmula é do tipo And, podemos
usar isinstance(f, And), e assim por diante . . . )

### Função Auxiliar

In [2]:
# Função auxiliar para detectar caso padrão
def isInstanceOfAndOrEqualsXOr(f) :
    return (isinstance(f, And) or
            isinstance(f, Or) or
            isinstance(f, Equals) or
            isinstance(f, XOr))

### Função Principal

In [150]:
def subFormula(f, seen = None):

    # Apenas para evitar duplicatas no vetor de saída.
    if seen is None:
        seen = set()
    if f in seen:
        return []

    seen.add(f)
    
    # Caso base: retorna lista com átomo
    if isinstance(f, Atom):
        return [f]
    # Caso padrão: retorna fórmula atual + chamada recursiva para subfórmulas maximais
    if isInstanceOfAndOrEqualsXOr(f):
        return [f] + sum([subFormula(sub) for sub in f.subformulas], [])
    # Caso neg: apenas uma subfórmula
    if isinstance(f, Neg):
        return [f] + subFormula(f.subformula)
    # Caso implies: acesso das subfórmulas via .left e .right
    if isinstance(f, Implies):
        return [f] + subFormula(f.left) + subFormula(f.right)

### Exemplo Teste

In [151]:
lista = []
n = 3
for j in range(n):
    for k in range(j + 1, n):
        lista.append(Neg(And(Atom('p1,' + str(j)), Atom('p1,' + str(k)))))

f = And(*lista)

In [152]:
for sub in subFormula(f):
    print(sub)

~(p1,0 & p1,1) & ~(p1,0 & p1,2) & ~(p1,1 & p1,2)
~(p1,0 & p1,1)
p1,0 & p1,1
p1,0
p1,1
~(p1,0 & p1,2)
p1,0 & p1,2
p1,0
p1,2
~(p1,1 & p1,2)
p1,1 & p1,2
p1,1
p1,2


# Tarefa 2 - Escreva a função (recursiva) formRand(n) que gera uma fórmula recursiva com n nós.

In [153]:
import random

def chooseRandomAtom(k) :
    return (Atom('p' + str(random.randint(1, k))))

operators = ['AND', 'OR', 'NEG']

# Adicionei um k opcional para determinar a pool de literais da fórmula. Por padrão, k = n
def formRand(n, k = None):
    if k is None:
        k = n
        
    # Fórmula com um nó só pode ser literal
    if (n == 1): 
        return chooseRandomAtom(k)
        
    # Fórmula com dois nós só pode ser negação
    if (n == 2):
        return Neg(formRand(n-1, k))

    # Para três ou mais nós, podemos aleatorizar o operador
    op = random.choice(operators)
    if (op == 'NEG'):
        return Neg(formRand(n-1, k))

    # Tenho (n - 1) nós disponíveis para dividir na subárvore esquerda e direita.

    # Tamanho esquerdo tem no máximo (n - 2) nós para garantir que árvore direita possui pelo menos um nó.
    left_size = random.randint(1, n - 2)
    # subárvore direita possui nós que sobraram
    right_size = (n - 1) - left_size

    # Para adicionar mas fórmulas (XOr, Implies, ...) seria só aumentar a lista de operadores 
    # e adicionar um if para cada caso aqui. 
    if (op == 'AND'):
        return And(formRand(left_size, k), formRand(right_size, k))             
    if (op == 'OR'):
        return Or(formRand(left_size, k), formRand(right_size, k))    

In [158]:
print(formRand(10, 3))

~(p1 | (p1 | p2)) & (p3 | p1)


# Tarefa 3 - Escreva a função fncon(f) que, dada uma fórmula, produz outra na forma normal conjuntiva (um objeto do tipo And cujas subfórmulas são do tipo Or que por sua vez possui apenas literais como subformulas).

# Tarefa 4 - Escreva a função clausulas(f) que retorna uma lista de listas de inteiros representando as cláusulas da fórmula obtida na tarefa anterior na forma normal conjuntiva